# Create Recipe Programatically

This notebook uses the Poetry-managed environment. All dependencies from `pyproject.toml` are available.


In [ ]:
# Verify we're using the correct environment
import sys
print(f"Python: {sys.executable}")


In [ ]:
from femtum_sdk.api.contracts.v1.hardware.health_pb2 import HealthRequest
from femtum_sdk.api import DatastoreApi, HardwareApi

datastore_host = "localhost:5208"
hardware_host = "localhost:5210"

with DatastoreApi(host=datastore_host) as datastore_api:
    datastore_health_response = datastore_api.health.GetHealth(HealthRequest())
    print("Datastore Health:")
    print(datastore_health_response)

with HardwareApi(host=hardware_host) as hardware_api:
    hardware_health_response = hardware_api.health.GetHealth(HealthRequest())
    print("Hardware Health:")
    print(hardware_health_response)


## Create Single Point Recipe

First get camera size to be assigned to recipe

In [ ]:
from femtum_sdk.api.contracts.v1.dto.camera_pb2 import CameraInfo
from femtum_sdk.api.contracts.v1.hardware.camera_pb2 import ReadCameraInfoRequest, ResetROIRequest

with HardwareApi(host=hardware_host) as hardware_api:
    hardware_api.laser_head_camera.ResetROI(ResetROIRequest())
    camera_info: CameraInfo = hardware_api.laser_head_camera.ReadCameraInfo(ReadCameraInfoRequest())
    context_width: int = camera_info.MaxFrameSize.Width
    context_height: int = camera_info.MaxFrameSize.Height

print(f"{context_width}, {context_height}")

In [ ]:
from femtum_sdk.api.contracts.v1.dto.geometry_pb2 import Attribute, Geometry, Point, Shape
from femtum_sdk.api.contracts.v1.dto.recipe_pb2 import LaserParameter
from femtum_sdk.api.contracts.v1.storage.recipe_pb2 import CreateRecipeRequest

with DatastoreApi(host=datastore_host) as datastore_api:
    single_point_recipe = datastore_api.recipe.Create(CreateRecipeRequest(
        Name="Single Point Marking Path", 
        Geometries=[
            Geometry(
                Type="laser_marking",
                Attributes=[
                    Attribute(
                        Key="marking_type",
                        Value="single_pulse",
                    )
                ],
                Shape=Shape(
                    Point=Point(
                        X=context_width/2,
                        Y=context_height/2,
                    )
                ),
            )
        ],
        Attributes=[
            Attribute(
                Key="context.width",
                Value=f"{context_width}",
            ),
            Attribute(
                Key="context.height",
                Value=f"{context_height}",
            ),
        ],
        LaserParameters=[
            LaserParameter(Key="pulse_train_laser_triggering_frequency_khz", Value="2732", Unit="kHz"),
            LaserParameter(Key="single_pulse_laser_triggering_frequency_khz", Value="2732", Unit="kHz"),
        ]
    ))


## Waveguide

In [ ]:
from femtum_sdk.api.contracts.v1.dto.geometry_pb2 import Attribute, Geometry, LineString, Point, Shape
from femtum_sdk.api.contracts.v1.dto.recipe_pb2 import LaserParameter
from femtum_sdk.api.contracts.v1.storage.recipe_pb2 import CreateRecipeRequest

with DatastoreApi(host=datastore_host) as datastore_api:
    single_point_recipe = datastore_api.recipe.Create(CreateRecipeRequest(
        Name="Single Linestring Waveguide", 
        Geometries=[
            Geometry(
                Type="waveguide",
                Attributes=[
                    Attribute(
                        Key="marking_type",
                        Value="single_pulse",
                    )
                ],
                Shape=Shape(
                    LineString=LineString(
                        Points=[
                            Point(
                                X=context_width/2,
                                Y=context_height/2,
                            ),
                            Point(
                                X=context_width/2 + 100,
                                Y=context_height/2 + 100,
                            )
                        ]
                    )
                ),
            )
        ],
        Attributes=[
            Attribute(
                Key="context.width",
                Value=f"{context_width}",
            ),
            Attribute(
                Key="context.height",
                Value=f"{context_height}",
            ),
        ],
        LaserParameters=[
            LaserParameter(Key="pulse_train_laser_triggering_frequency_khz", Value="2732", Unit="kHz"),
            LaserParameter(Key="single_pulse_laser_triggering_frequency_khz", Value="2732", Unit="kHz"),
        ]
    ))

## Cleanup

Connections are closed automatically when each `with` block exits, so no manual cleanup is required.